In [1]:
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
)
from sklearn.preprocessing import StandardScaler

# Loading Processed Dataset
bundle = joblib.load("../backend/models/titanic/titanic_data_bundle.pkl")

X_train = bundle["X_train"]
X_test = bundle["X_test"]
y_train = bundle["y_train"]
y_test = bundle["y_test"]

encoders = bundle["encoders"]
power_transformer = bundle["power_transformer"]
scaler = bundle["scaler"]
selectors = bundle["selected_features"]
title_medians = bundle["title_medians"]
age_median = bundle["age_median"]
embarked_mode = bundle["embarked_mode"]

print("Success!")
print(f"X_train size: {X_train.shape}, X_test size: {X_test.shape}")

Success!
X_train size: (878, 5), X_test size: (179, 5)


In [2]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

In [3]:
# K-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr_model, X_train, y_train, cv=cv, scoring="accuracy")

print("K-Fold (K=5) Validation Scores:", cv_scores)
print("Average CV Validation: {:.4f} (+/- {:.4f})".format(cv_scores.mean(), cv_scores.std() * 2))

K-Fold (K=5) Validation Scores: [0.78977273 0.79545455 0.79545455 0.82857143 0.76      ]
Average CV Validation: 0.7939 (+/- 0.0436)


In [4]:
# Training the Model
lr_model.fit(X_train, y_train)
print("\nModel is trained!")


Model is trained!


In [5]:
# Train Set Performance (For Overfitting)
y_train_pred = lr_model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)

print("=" * 40)
print("          TRAIN SET PERFORMANCE          ")
print("=" * 40)
print(f"Train Accuracy: {train_accuracy:.4f}\n")
print("Train Classification (Precision, Recall, F1-Score):")
print(classification_report(y_train, y_train_pred))

# Test Set Performance
y_test_pred = lr_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("=" * 40)
print("          TEST SET PERFORMANCE           ")
print("=" * 40)
print(f"Test Accuracy: {test_accuracy:.4f}\n")
print("Test Classification (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_test_pred))

          TRAIN SET PERFORMANCE          
Train Accuracy: 0.7950

Train Classification (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

           0       0.82      0.76      0.79       439
           1       0.78      0.83      0.80       439

    accuracy                           0.79       878
   macro avg       0.80      0.79      0.79       878
weighted avg       0.80      0.79      0.79       878

          TEST SET PERFORMANCE           
Test Accuracy: 0.6369

Test Classification (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

           0       0.67      0.81      0.73       110
           1       0.54      0.36      0.43        69

    accuracy                           0.64       179
   macro avg       0.61      0.59      0.58       179
weighted avg       0.62      0.64      0.62       179



/home/gokay/PycharmProjects/FastApiProject/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [6]:
# Saving the Model
joblib.dump(lr_model, "../backend/models/titanic/logistic_regression_model.pkl")
print("Trained model is successfully saved!")

Trained model is successfully saved!


In [7]:
# --- Feature Importance (Coefficients) ---
print("=" * 45)
print("             FEATURE COEFFICIENTS              ")
print("=" * 45)
feature_importance = pd.DataFrame(
    {"Feature": selectors, "Coefficient": lr_model.coef_[0]}
)
feature_importance["Abs_Coefficient"] = feature_importance["Coefficient"].abs()
feature_importance = feature_importance.sort_values(by="Abs_Coefficient", ascending=False)
print(feature_importance[["Feature", "Coefficient"]])

             FEATURE COEFFICIENTS              
        Feature  Coefficient
3  Cabin_Deck_U    -0.695238
0      Title_Mr    -0.662384
1    Sex_female     0.649493
2          Fare     0.246568
4     Title_Mrs     0.014137


In [8]:
# --- 1. Load Raw Dataset ---
df = pd.read_csv("../datasets/titanic.csv")

# --- 2. Basic Data Cleaning & Preprocessing ---
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

# Fill missing values with basic statistics
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# One-Hot Encode categorical variables
df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)

# --- 3. Split Features and Target ---
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standard scaling for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 4. Model Training ---
lr_raw = LogisticRegression(max_iter=1000, random_state=42)
lr_raw.fit(X_train_scaled, y_train)

# --- 5. Train Set Performance ---
y_train_pred = lr_raw.predict(X_train_scaled)
y_train_proba = lr_raw.predict_proba(X_train_scaled)[:, 1]
train_accuracy = accuracy_score(y_train, y_train_pred)

print("=" * 45)
print("              TRAIN SET PERFORMANCE              ")
print("=" * 45)
print(f"Train Accuracy: {train_accuracy:.4f}")
print("Train Classification Report:")
print(classification_report(y_train, y_train_pred))

# --- 6. Test Set Performance ---
y_test_pred = lr_raw.predict(X_test_scaled)
y_test_proba = lr_raw.predict_proba(X_test_scaled)[:, 1]
test_accuracy = accuracy_score(y_test, y_test_pred)

print("=" * 45)
print("              TEST SET PERFORMANCE               ")
print("=" * 45)
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

              TRAIN SET PERFORMANCE              
Train Accuracy: 0.8076
Train Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.87      0.85       439
           1       0.77      0.71      0.74       273

    accuracy                           0.81       712
   macro avg       0.80      0.79      0.79       712
weighted avg       0.81      0.81      0.81       712

              TEST SET PERFORMANCE               
Test Accuracy: 0.8045

Test Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

